# Анализ loss landscape

Ноутбук для воспроизведения ключевых графиков и таблиц (данные из `data/processed/results/`).

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "config.py").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SRC))

import torch

import config
from metrics import compute_metrics
from plotting import (
    plot_comparison,
    plot_metrics_vs_radius,
    plot_surface_3d,
    plot_surface_contour,
)
from quadratic_fit import (
    fit_diagonal_quadratic,
    fit_full_quadratic,
    fit_hessian_quadratic,
)
from model import build_model
from surface import get_val_loader

torch.manual_seed(config.SEED)
np.random.seed(config.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.SEED)

MODEL_NAME = "model1"  # или model2
R = config.RADIUS_DEFAULT

In [ ]:
npz_path = ROOT / "data" / "processed" / "results" / f"surface_{MODEL_NAME}_r{R}.npz"
data = np.load(npz_path)
alpha = data["alpha_grid"]
beta = data["beta_grid"]
f = data["f"]
theta_flat = data["theta_flat"]
d1_flat = data["d1_flat"]
d2_flat = data["d2_flat"]

In [ ]:
plot_surface_3d(
    alpha,
    beta,
    f,
    title=f"Loss surface ({MODEL_NAME})",
    filename=f"nb_surface_3d_{MODEL_NAME}.png",
)
plot_surface_contour(
    alpha,
    beta,
    f,
    title=f"Loss contour ({MODEL_NAME})",
    filename=f"nb_surface_contour_{MODEL_NAME}.png",
)

In [ ]:
p1, f1 = fit_full_quadratic(alpha, beta, f)
p2, f2 = fit_diagonal_quadratic(alpha, beta, f)

device = torch.device(config.DEVICE)
model = build_model(MODEL_NAME).to(device)
_ckpt_path = ROOT / "data" / "processed" / "checkpoints" / f"{MODEL_NAME}_final.pth"
try:
    ckpt = torch.load(_ckpt_path, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(_ckpt_path, map_location=device)
model.load_state_dict(ckpt["state_dict"])
loader = get_val_loader()
p3, f3 = fit_hessian_quadratic(
    model, loader, device, alpha, beta, theta_flat, d1_flat, d2_flat
)

print("full_quadratic:", p1)
print("diagonal_quadratic:", p2)
print("hessian_quadratic:", p3)

In [ ]:
rows = []
for label, fh in [
    ("full_quadratic", f1),
    ("diagonal_quadratic", f2),
    ("hessian_quadratic", f3),
]:
    m = compute_metrics(f, fh)
    rows.append({"approx_type": label, **m})
metrics_df = pd.DataFrame(rows)
try:
    from IPython.display import display

    display(
        metrics_df.style.format(
            {"RMSE": "{:.4e}", "L_inf": "{:.4e}", "RelRMSE": "{:.4e}"}
        )
    )
except Exception:
    print(metrics_df.to_string(index=False))

In [ ]:
plot_comparison(
    alpha,
    beta,
    f,
    f1,
    f2,
    f3,
    filename=f"nb_comparison_{MODEL_NAME}.png",
)

In [ ]:
csv_path = ROOT / "data" / "processed" / "results" / f"metrics_vs_radius_{MODEL_NAME}.csv"
df_rad = pd.read_csv(csv_path)
display(df_rad)
plot_metrics_vs_radius(df_rad, filename=f"nb_metrics_vs_radius_{MODEL_NAME}.png")